In [10]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from datasets import Dataset

# Load the PCL dataset
def load_data():
    df_pcl = pd.read_csv("dontpatronizeme_pcl.tsv", sep="\t", header=None, skiprows=4,
                         names=["par_id", "art_id", "keyword", "country", "text", "label"])

    df_pcl = df_pcl.dropna(subset=['text'])
    
    train_lbl = pd.read_csv("train_semeval_parids-labels.csv")
    dev_lbl = pd.read_csv("dev_semeval_parids-labels.csv")
    
    # Process labels: y=1 if at least one annotator marked PCL
    train_lbl["label_vec"] = train_lbl["label"].apply(ast.literal_eval)
    dev_lbl["label_vec"] = dev_lbl["label"].apply(ast.literal_eval)
    train_lbl["y"] = train_lbl["label_vec"].apply(lambda v: int(sum(v) > 0))
    dev_lbl["y"] = dev_lbl["label_vec"].apply(lambda v: int(sum(v) > 0))
    
    train_df = df_pcl.merge(train_lbl[["par_id", "y"]], on="par_id", how="inner")
    dev_df = df_pcl.merge(dev_lbl[["par_id", "y"]], on="par_id", how="inner")
    
    return train_df, dev_df

train_df, dev_df = load_data()
print(f"Data Loaded. Train size: {len(train_df)}, Dev size: {len(dev_df)}")
dev_df.head()

Data Loaded. Train size: 8375, Dev size: 2093


,par_id,art_id,keyword,country,text,label,y
0,107,@@16900972,homeless,ke,"His present "" chambers "" may be quite humble ,...",3,1
1,149,@@1387882,disabled,us,Krueger recently harnessed that creativity to ...,2,1
2,151,@@19974860,poor-families,in,10:41am - Parents of children who died must ge...,3,1
3,154,@@20663936,disabled,ng,When some people feel causing problem for some...,4,1
4,157,@@21712008,poor-families,ca,We are alarmed to learn of your recently circu...,4,1


In [11]:
def tokenize_fn(examples, tokenizer):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, pos_label=1)
    return {"f1": f1}

# Convert pandas to HuggingFace Dataset format
train_ds = Dataset.from_pandas(train_df[['text', 'y']].rename(columns={'y': 'label'}))
dev_ds = Dataset.from_pandas(dev_df[['text', 'y']].rename(columns={'y': 'label'}))

In [12]:
# Analysis: Count missing values in the text column
print("Missing values in Train:", train_df['text'].isnull().sum())
print("Missing values in Dev:", dev_df['text'].isnull().sum())

# If the count is > 0, see which rows are affected
print("\nRows with missing text in Dev:")
display(dev_df[dev_df['text'].isnull()])

Missing values in Train: 0
Missing values in Dev: 0

Rows with missing text in Dev:


,par_id,art_id,keyword,country,text,label,y


In [16]:
import torch
if torch.backends.mps.is_available():
    print("Apple Silicon GPU (MPS) is available.")
elif torch.cuda.is_available():
    print("NVIDIA GPU (CUDA) is available.")
else:
    print("No GPU found. Training on CPU.")

Apple Silicon GPU (MPS) is available.


# Baseline

In [ ]:
MODEL_NAME_BASE = "roberta-base"
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME_BASE)
tokenized_train_base = train_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)
tokenized_dev_base = dev_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)

training_args_base = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",      # When to evaluate
    save_strategy="epoch",      # MUST match eval_strategy
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True, # Now this works!
    metric_for_best_model="f1",
)

trainer_base = Trainer(
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_BASE, num_labels=2),
    args=training_args_base,
    train_dataset=tokenized_train_base,
    eval_dataset=tokenized_dev_base,
    compute_metrics=compute_metrics,
)

trainer_base.train()

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'use_mps_device'

In [ ]:
import torch

# 1. Set the device to MPS
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

MODEL_NAME_BASE = "roberta-base"
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME_BASE)

# 2. Prepare datasets (Mapping remains the same as it happens on CPU/RAM)
tokenized_train_base = train_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)
tokenized_dev_base = dev_ds.map(lambda x: tokenize_fn(x, tokenizer_base), batched=True)

# 3. Load model and move it to the MPS device
model_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_BASE, num_labels=2)
model_base.to(device)

training_args_base = TrainingArguments(
    output_dir="./results_baseline",
    eval_strategy="epoch",      
    save_strategy="epoch",      
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True, 
    metric_for_best_model="f1",
    # Remove any use_mps_device=True as it is deprecated in newer versions
)

# 4. Initialize Trainer
trainer_base = Trainer(
    model=model_base,  # This model is now on the MPS device
    args=training_args_base,
    train_dataset=tokenized_train_base,
    eval_dataset=tokenized_dev_base,
    compute_metrics=compute_metrics,
)

# 5. Execute Training
trainer_base.train()

Using device: mps


Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2093 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


# Improved Model

In [ ]:
MODEL_NAME_IMP = "microsoft/deberta-v3-base"
tokenizer_imp = AutoTokenizer.from_pretrained(MODEL_NAME_IMP)
tokenized_train_imp = train_ds.map(lambda x: tokenize_fn(x, tokenizer_imp), batched=True)
tokenized_dev_imp = dev_ds.map(lambda x: tokenize_fn(x, tokenizer_imp), batched=True)

# Calculate weights to penalize the model more for missing the PCL class
class_weights = torch.tensor([
    len(train_df) / (2 * len(train_df[train_df.y == 0])),
    len(train_df) / (2 * len(train_df[train_df.y == 1]))
]).to("cuda" if torch.cuda.is_available() else "cpu")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.float())
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args_imp = TrainingArguments(
    output_dir="./results_improved",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer_imp = WeightedTrainer(
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_IMP, num_labels=2),
    args=training_args_imp,
    train_dataset=tokenized_train_imp,
    eval_dataset=tokenized_dev_imp,
    compute_metrics=compute_metrics,
)

trainer_imp.train()

# Compare Models

In [ ]:
def get_report(trainer, tokenized_data, title):
    print(f"\n--- {title} Results ---")
    preds = trainer.predict(tokenized_data)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    print(classification_report(preds.label_ids, pred_labels, target_names=["No PCL", "PCL"]))

get_report(trainer_base, tokenized_dev_base, "RoBERTa Baseline")
get_report(trainer_imp, tokenized_dev_imp, "DeBERTa Improved")